# 02 — Build the Prompt Yourself. Then Let the Architect Fix It.

**The pitch.** You already have a prompt — or a rough idea of one. `PromptArchitect.parse()` diagnoses it section by section, scores each one, and tells you exactly what's weak. `improve()` rewrites only the weak sections and shows you the diff. `GuidanceOptimizer` upgrades vague rules into binding, testable constraints. Every step is measurable.

**What you'll see in this notebook:**
- Start with a realistic but weak manually-written prompt
- Score it before doing anything
- `parse()` — map it to the 9 sections, expose what's missing
- Manual upgrade — apply thinking strategies and tighten constraints
- `improve()` — let the architect fix what's still weak
- `GuidanceOptimizer` — upgrade vague rules to binding modals
- `diff_report()` — see exactly what changed and why
- Final score comparison: v0 → v1 (manual) → v2 (architect-improved)
- Optional **quality controls** on `Constraints` after `improve()` (0.11+; **GUARD RAILS** text in `assemble()` from **0.11.1**): inspect verbosity, tone, answer-first, forbidden phrases, self-check

---
> **Research grounding.** The improvement process is grounded in two bodies of research:
> 1. **Iterative prompt refinement**: Small, targeted edits to specific sections outperform wholesale rewrites (*Pryzant et al. 2023, "Automatic Prompt Optimization with 'Gradient Descent'"*). `improve()` implements this — it surgically rewrites only the sections below threshold.
> 2. **Linguistic specificity**: Vague rules degrade constraint following. Binding modals (must/never) produce significantly more consistent adherence than hedged language (should/try) (*Mizrahi et al. 2023, "State of What Art?"*). `GuidanceOptimizer` applies this finding directly — it rewrites every vague rule into a binding, testable constraint.

## Install and setup

In [1]:
#%pip install -q -U "mycontext-ai>=0.11.1"
import os
import sys
import warnings
from pathlib import Path


warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from dotenv import load_dotenv
from IPython.display import display_markdown

project_root = Path.cwd()
# adjust as needed
env_path = project_root / "content-series" / ".env"


load_dotenv(env_path)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

def md(s):
    display_markdown(s,raw=True)

import mycontext


md(f"**mycontext-ai** `{getattr(mycontext, '__version__', '?')}`")



**mycontext-ai** `0.11.1`

## The starting point: a realistic weak prompt

This is the kind of prompt that gets written in 10 minutes and ships. It's not terrible — it has a role and a task. But it leaves enormous performance on the table.

In [2]:
weak_prompt = """You are a helpful data analyst.
Analyze the customer support tickets provided.
Try to be accurate and avoid making assumptions.
Summarize the key issues and sentiment.
Be concise and helpful."""

md("Weak prompt:")
md("-" * 50)
md(weak_prompt)
md("-" * 50)
md(f"Length: {len(weak_prompt)} chars | Sections covered: ~2 of 9")


Weak prompt:

--------------------------------------------------

You are a helpful data analyst.
Analyze the customer support tickets provided.
Try to be accurate and avoid making assumptions.
Summarize the key issues and sentiment.
Be concise and helpful.

--------------------------------------------------

Length: 191 chars | Sections covered: ~2 of 9

## Step 1 — Baseline score (before anything)

Score it with `QualityMetrics` now, so we have a number to beat. No API key needed.

In [3]:
from mycontext import Context, Guidance, Directive
from mycontext.intelligence import QualityMetrics

# Wrap in a Context for scoring
v0_ctx = Context(
    guidance=Guidance(
        role="helpful data analyst",
        rules=["Try to be accurate", "Avoid making assumptions", "Be concise and helpful"],
    ),
    directive=Directive(content="Analyze customer support tickets. Summarize key issues and sentiment."),
)

qm = QualityMetrics(mode="heuristic")
v0_score = qm.evaluate(v0_ctx)

md(f"Baseline (v0) quality: {v0_score.overall * 100:.0f}/100")
md("\nDimension breakdown:")
for _dk, _dv in v0_score.dimensions.items():
    _pct = _dv * 100.0
    _filled = int(_pct / 10)
    _bar = '█' * _filled + '░' * (10 - _filled)
    md(f"  {_dk.name.title():<25} {_bar} {_pct:.0f}")


Baseline (v0) quality: 48/100


Dimension breakdown:

  Clarity                   █████░░░░░ 58

  Completeness              ████░░░░░░ 45

  Specificity               █░░░░░░░░░ 10

  Relevance                 ████░░░░░░ 45

  Structure                 █████░░░░░ 55

  Efficiency                █████████░ 90

## Step 2 — `parse()`: see exactly what the prompt has and what it's missing

In [4]:
from mycontext.intelligence import PromptArchitect

architect = PromptArchitect(provider="openai")
parsed = architect.parse(weak_prompt)

md("=== Section analysis ===")
md(parsed)


=== Section analysis ===

{"role":"a helpful data analyst","goal":"Analyze the customer support tickets provided.","rules":[],"style":null,"reasoning":null,"examples":[],"output_contract":null,"guard_rails":[],"task":"Be concise and helpful."}

In [5]:
# Specific problems parse() will find in our weak prompt:
problems = [
    ("rules",      "WEAK",    "'Try to be accurate' — 'try' is a hedge, not a binding constraint (should be 'must')"),
    ("rules",      "WEAK",    "'Avoid making assumptions' — bare negation with no fallback phrase"),
    ("rules",      "WEAK",    "'Be concise' — not testable. What is concise? No criterion."),
    ("reasoning",  "MISSING", "No thinking strategy — model uses default reasoning, which varies"),
    ("examples",   "MISSING", "No few-shot demonstrations"),
    ("output_contract", "MISSING", "No output format specified — could return anything"),
    ("guard_rails", "MISSING", "No uncertainty fallback — model will hallucinate rather than say 'not found'"),
    ("goal",       "MISSING", "No explicit mission statement beyond task description"),
]

md(f"{'Section':<16} {'Status':<10} {'Problem'}")
md("-" * 90)
for section, status, problem in problems:
    md(f"{section:<16} {status:<10} {problem}")


Section          Status     Problem

------------------------------------------------------------------------------------------

rules            WEAK       'Try to be accurate' — 'try' is a hedge, not a binding constraint (should be 'must')

rules            WEAK       'Avoid making assumptions' — bare negation with no fallback phrase

rules            WEAK       'Be concise' — not testable. What is concise? No criterion.

reasoning        MISSING    No thinking strategy — model uses default reasoning, which varies

examples         MISSING    No few-shot demonstrations

output_contract  MISSING    No output format specified — could return anything

guard_rails      MISSING    No uncertainty fallback — model will hallucinate rather than say 'not found'

goal             MISSING    No explicit mission statement beyond task description

## Step 3 — Manual upgrade (v1): fix the rules, add strategy, add output contract

Apply what we know from the research: binding modals, testable rules, thinking strategy in the middle zone, output contract, fallback phrase.

In [6]:
from mycontext import Constraints

v1_ctx = Context(
    guidance=Guidance(
        role="Senior customer intelligence analyst specialising in support ticket analysis",
        goal="Surface actionable patterns in support tickets that product and ops teams can act on",
        rules=[
            # Binding modals — replacing 'try' and 'avoid'
            "Must classify each ticket by issue type using the taxonomy: billing, technical, UX, policy, other",
            "Must assign a sentiment score (positive/neutral/negative) with a one-sentence evidence quote",
            "Must flag any ticket with urgency ≥4/5 explicitly as HIGH PRIORITY",
            # Grounding protocol
            "Every classification must cite the specific ticket phrase that supports it",
            # Positive redirect replacing bare negation
            "If the ticket contains insufficient information to classify, state: 'INSUFFICIENT DATA — [missing field]'",
        ],
        style="Precise, structured, written for a product manager audience",
    ),
    directive=Directive(content="Analyze the customer support tickets below and return the structured analysis"),
    constraints=Constraints(
        format_rules=[
            "Return a JSON array — one object per ticket",
            "Fields: ticket_id, issue_type, sentiment, urgency (1-5), evidence_quote, priority_flag",
        ],
        must_not_include=["stock phrases like 'it seems', 'appears to be', 'might be'"],
    ),
    # Research-backed: analytical archetype in middle zone
    thinking_strategy="verify",  # Self-reflection: review classification before returning
    research_flow=True,
)

v1_score = qm.evaluate(v1_ctx)
md(f"v1 (manual upgrade) quality: {v1_score.overall * 100:.0f}/100")
md(f"Improvement over baseline:   +{(v1_score.overall - v0_score.overall) * 100:.0f} points")


v1 (manual upgrade) quality: 81/100

Improvement over baseline:   +33 points

## Step 4 — `improve()`: let the architect fix what's still weak (v2)

In [7]:
improved = architect.improve(weak_prompt)
v2_ctx = improved.improved_context
v2_score = qm.evaluate(v2_ctx)

md(f"v2 (architect-improved) quality: {v2_score.overall * 100:.0f}/100")
md(f"Improvement over baseline:       +{(v2_score.overall - v0_score.overall) * 100:.0f} points")
md("\n=== Improved prompt ===")
md((str(v2_ctx.assemble())[:1500-3] + '...' if len(str(v2_ctx.assemble())) > 1500 else str(v2_ctx.assemble())))


v2 (architect-improved) quality: 75/100

Improvement over baseline:       +26 points


=== Improved prompt ===

## ROLE

You are a data analyst. Scope: Limit to customer support tickets analysis. Do not deviate from this focus.

## GOAL

**Your mission:** Summarize key issues and sentiment from customer support tickets — accomplish this fully.

## RULES

**You MUST follow these rules at all times:**
  1. Every summary must include at least 3 key issues identified in the tickets.
  2. Every sentiment analysis must categorize sentiment as positive, negative, or neutral.
  3. Every finding must include supporting evidence from the ticket data.
  4. Every claim must be grounded in the provided source material. If absent, state: 'Not found in the provided material.'

## STYLE

**Tone & voice:** Formal, concise, general public. Omit jargon without definition and preamble or greeting.

## EXAMPLES

Learn from these examples of expected input → output:

**Example 1:**
Input: Customer ticket: 'I am unhappy with the service I received.'
Output: KEY FINDING: Negative sentiment regarding service. EVIDENCE: Customer explicitly states unhappiness. RECOMMENDED ACTION: Investigate service quality issues.

**Example 2:**
Input: Customer ticket: 'The product works great, but the delivery was late.'
Output: KEY FINDING: Mixed sentiment with positive product feedback and negative delivery experience. EVIDENCE: Customer praises product but mentions late delivery. RECOMMENDED ACTION: Improve delivery timelines.

**Example 3:**
Input: Customer ticket: 'Not found in the provided material.'
Output: KEY FIND...

## Quality controls after `improve()` (0.11.1+)

The same optional **Constraints** quality fields (`verbosity`, `communication_posture`, `answer_first`, `forbidden_phrases`, `self_check`) can be populated when the architect rewrites your prompt. With **mycontext-ai ≥ 0.11.1**, `improved.improved_context.assemble()` includes them under **## GUARD RAILS** (same copy as `Constraints.render_quality_segments()`). They remain **user-overridable** on the context before execution.

In [8]:
qc = v2_ctx.constraints
fields = (
    "verbosity",
    "communication_posture",
    "answer_first",
    "forbidden_phrases",
    "self_check",
)
lines = []
for name in fields:
    val = getattr(qc, name, None)
    if val is not None:
        lines.append(f"- **{name}:** `{val!r}`")
md(
    "**Quality fields on the improved context** (when present):\n\n"
    + (
        "\n".join(lines)
        if lines
        else "_None set after this improve() — API path may omit them; confirm mycontext-ai ≥ 0.11.1._"
    )
)

**Quality fields on the improved context** (when present):

- **verbosity:** `'standard'`
- **communication_posture:** `'direct'`
- **answer_first:** `False`
- **self_check:** `['Did I identify at least 3 key issues from the tickets?', 'Is the sentiment accurately categorized as positive, negative, or neutral?', 'Is every claim supported by evidence from the ticket data?']`

## Step 5 — `diff_report()`: see exactly what changed and why

In [9]:
md(improved.diff_report())


── SECTION DIFF ──────────────────────────────────────────

[STRENGTHENED]  § ROLE
  BEFORE: a helpful data analyst
  AFTER:  You are a data analyst. Scope: Limit to customer support tickets analysis. Do not deviate from this focus.
  WHY:    Existing content was weak or incomplete; upgraded per: Formula: 'You are' + seniority + domain + optional context/setting.

[STRENGTHENED]  § GOAL
  BEFORE: Analyze the customer support tickets provided.
  AFTER:  Your mission: Summarize key issues and sentiment from customer support tickets — accomplish this fully.
  WHY:    Existing content was weak or incomplete; upgraded per: Formula: 'Your mission: [specific achievement] — accomplish this fully.'

[ADDED]  § RULES
  AFTER:  Every summary must include at least 3 key issues identified in the tickets.; Every sentiment analysis must categorize sentiment as positive, negative, or neutral.; Every finding must include supportin
  WHY:    Section was absent; added using 9-section principle: Rules are guarantees, not preferences. BAD: 'Try to be thorough.' GOOD: 'Every finding must include a code location, severity, and remediation step.'

[ADDED]  § STYLE
  AFTER:  Formal, concise, general public. Omit jargon without definition and preamble or greeting.
  WHY:    Section was absent; added using 9-section principle: Specify formality (formal/casual/technical), pace (concise/thorough/step-by-step), audience (senior engineer/executive/general public).

[ADDED]  § REASONING
  AFTER:  analytical
  WHY:    Section was absent; added using 9-section principle: Read the user's GOAL and TASK. Choose reasoning strategy slugs that fit the *question*, not a default.

[ADDED]  § EXAMPLES
  AFTER:  {'input': "Customer ticket: 'I am unhappy with the service I received.'", 'output': 'KEY FINDING: Negative sentiment regarding service. EVIDENCE: Customer explicitly states unhappiness. RECOMMENDED AC
  WHY:    Section was absent; added using 9-section principle: 2-5 representative examples. Fewer than 2 = too much ambiguity; more than 5 = token waste.

[ADDED]  § OUTPUT CONTRACT
  AFTER:  Return ONLY a 3-part structure: 1) KEY FINDING (1 sentence), 2) EVIDENCE (2-3 bullets), 3) RECOMMENDED ACTION (1 sentence).
  WHY:    Section was absent; added using 9-section principle: Start with 'Return ONLY' — these two words reliably suppress preamble.

[ADDED]  § GUARD RAILS
  AFTER:  Omit speculation.; Omit hedging language: probably, might, could be.; Omit stock AI phrases: in today's rapidly evolving, delve into.
  WHY:    Section was absent; added using 9-section principle: POSITIVE REDIRECTS only. BAD: 'Do not hallucinate.' GOOD: 'Every claim must be grounded in the source material. If absent, state: "Not found in the provided material."'

[STRENGTHENED]  § TASK
  BEFORE: Be concise and helpful.
  AFTER:  Analyze the provided customer support tickets and summarize key issues and sentiment.
  WHY:    Existing content was weak or incomplete; upgraded per: Place LAST (recency zone — highest attention when generation begins).

──────────────────────────────────────────────────────

## Step 6 — `GuidanceOptimizer`: the rules specialist

`GuidanceOptimizer` is a focused tool for one specific problem: **vague rules**. Research shows that binding modals (must/never) produce significantly more consistent constraint following than hedged language (should/try). The optimizer rewrites every vague rule without touching anything else.

In [10]:
from mycontext.intelligence import GuidanceOptimizer
from mycontext import Guidance

# A realistic set of vague rules that survives most code reviews
vague_guidance = Guidance(
    role="Support analyst",
    rules=[
        "Be accurate and thorough",
        "Try to be helpful",
        "Avoid making things up",
        "Use good judgment when information is missing",
        "Be concise but complete",
    ],
)

optimizer = GuidanceOptimizer()

# audit() is heuristic — no API key needed
audit = optimizer.audit(vague_guidance)
md("=== Guidance audit (heuristic, no API key needed) ===")
md(audit)


=== Guidance audit (heuristic, no API key needed) ===

{"role_is_generic":false,"total_rules":5,"binding_rules":1,"weak_rules":[{"original":"Be accurate and thorough","issues":["vague_directive"],"rewritten":null,"action":"pending"},{"original":"Try to be helpful","issues":["suggestive_modal","vague_directive"],"rewritten":null,"action":"pending"},{"original":"Avoid making things up","issues":["too_short"],"rewritten":null,"action":"pending"},{"original":"Be concise but complete","issues":["vague_directive"],"rewritten":null,"action":"pending"}],"rule_strength_score":0}

In [11]:
optimized = optimizer.optimize(vague_guidance, provider="openai")

md("BEFORE (vague):")
for r in vague_guidance.rules:
    md(f"  ✗ {r}")

md("\nAFTER (binding + testable):")
for r in optimized.optimized_guidance.rules:
    md(f"  ✓ {r}")


BEFORE (vague):

  ✗ Be accurate and thorough

  ✗ Try to be helpful

  ✗ Avoid making things up

  ✗ Use good judgment when information is missing

...

  ✓ Every claim must include at least one verifiable source that supports the information provided.

  ✓ Every response must include at least one actionable suggestion or resource for the user.

  ✓ Every response must be based on verified information and include citations for any claims made.

  ✓ Use good judgment when information is missing

  ✓ Every response must contain all necessary information while being no longer than 150 words.

## Step 7 — Final comparison: v0 → v1 → v2

In [12]:
scores = [
    ("v0 — original weak prompt", v0_score.overall),
    ("v1 — manual upgrade (rules, strategy, schema)", v1_score.overall),
]
if 'v2_score' in dir():
    scores.append(("v2 — architect improve() on v0", v2_score.overall))

md(f"{'Version':<48} {'Score':>8}")
md("-" * 60)
for label, score in scores:
    pct = score * 100.0
    bar = '█' * int(pct / 5)
    md(f"{label:<48} {pct:>7.0f}/100  {bar}")

best = max(scores, key=lambda x: x[1])
md(f"\nBest: {best[0]} at {best[1] * 100:.0f}/100")
md(f"Total lift from v0: +{(best[1] - v0_score.overall) * 100:.0f} points")


Version                                             Score

------------------------------------------------------------

v0 — original weak prompt                             48/100  █████████

v1 — manual upgrade (rules, strategy, schema)         81/100  ████████████████

v2 — architect improve() on v0                        75/100  ██████████████


Best: v1 — manual upgrade (rules, strategy, schema) at 81/100

Total lift from v0: +33 points

## Summary

| What you learned | API | Key? |
|-----------------|-----|------|
| Score any Context before calling the model | `QualityMetrics(mode='heuristic').evaluate(ctx)` | No |
| Diagnose an existing prompt section by section | `PromptArchitect.parse(prompt)` | No |
| Rewrite only weak sections | `PromptArchitect.improve(prompt)` | Yes |
| See exactly what changed | `improved.diff_report()` | No |
| Audit vague rules | `GuidanceOptimizer.audit(guidance)` | No |
| Rewrite rules with binding modals | `GuidanceOptimizer.optimize(guidance)` | Yes |
| Upgrade manually with research-backed structure | `Context(research_flow=True, thinking_strategy='verify', ...)` | No |

**Next:** [03_smart_execute.ipynb](03_smart_execute.ipynb) — when you don't want to build or improve manually and just want the best answer to a question in one call.